# CSB-Style Toxic Release Tutorial (Deep Safety)

This notebook demonstrates a CSB-style toxic release workflow using Deep Safety's source, dispersion, toxic criteria, and effect models.

In [ ]:
from deepsafety.source_models import solve_source_model
from deepsafety.dispersion_service import solve_dispersion_model
from deepsafety.toxic_criteria import lookup_toxic_criteria
from deepsafety.effect_models import solve_effect_model

In [ ]:
source_inputs = {
    'source_subtype': 'hole',
    'duration_s': 300.0,
    'upstream_pressure_pa': 2_500_000.0,
    'downstream_pressure_pa': 101_325.0,
    'temperature_k': 298.15,
    'heat_capacity_ratio': 1.33,
    'molecular_weight_kg_kmol': 17.03,
    'hole_diameter_m': 0.006,
    'inventory_mass_kg': 1200.0,
}

source_result = solve_source_model('gas_release', source_inputs)
source_result

In [ ]:
dispersion_inputs = {
    'release_rate_kg_s': source_result['average_release_rate_kg_s'],
    'release_height_m': 1.5,
    'wind_speed_m_s': 2.0,
    'stability_class': 'F',
    'x_m': 500.0,
    'y_m': 0.0,
    'z_m': 1.5,
}

dispersion_result = solve_dispersion_model('gaussian_plume', dispersion_inputs)
dispersion_result

In [ ]:
criteria = lookup_toxic_criteria({
    'material': 'ammonia',
    'criteria': ['AEGL-2', 'ERPG-2', 'IDLH']
})
criteria

In [ ]:
effects_result = solve_effect_model('toxic_probit', {
    'concentration_kg_m3': dispersion_result['concentration_kg_m3'],
    'exposure_time_s': 600.0,
    'a': -14.3,
    'b': 2.3,
    'n': 2.0,
    'population_distribution': [
        {'id': 'near', 'label': 'Near Field', 'population': 120},
        {'id': 'mid', 'label': 'Mid Field', 'population': 280}
    ]
})
effects_result

## Suggested validation checks

- Compare realistic and worst-case source assumptions.
- Evaluate endpoint distances against AEGL/ERPG thresholds.
- Run a sensitivity sweep by varying release duration and wind speed.